In [ ]:
import requests
import xml.etree.ElementTree as ET
from typing import List, Dict, Any, Optional
import pandas as pd
import time
import os
import re
from tqdm import tqdm


def extract_unique_genes_with_communities(gene_sets: Dict[str, List[str]]) -> tuple:
    """Extract all unique genes and track which communities they belong to"""
    gene_to_communities = {}
    for community, genes in gene_sets.items():
        for gene in genes:
            if gene not in gene_to_communities:
                gene_to_communities[gene] = []
            gene_to_communities[gene].append(community)

    unique_genes = list(gene_to_communities.keys())
    print(f"  Extracted {len(unique_genes)} unique genes from {len(gene_sets)} gene sets")
    return unique_genes, gene_to_communities


def fetch_pmc_full_text(
    pmcid: str,
    email: str,
    api_key: Optional[str] = None,
) -> Optional[str]:
    """
    Fetch full-text XML from PubMed Central for a given PMCID and return
    the concatenated body text.  Returns None if unavailable.
    """
    try:
        params = {
            "db":      "pmc",
            "id":      pmcid,
            "rettype": "full",
            "retmode": "xml",
            "email":   email,
            "tool":    "AML_UniqueGeneResearch",
        }
        if api_key:
            params["api_key"] = api_key

        resp = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=params,
            headers={"User-Agent": f"AML_UniqueGeneResearch/1.0 (mailto:{email})"},
            timeout=30,
        )
        if not resp.ok:
            return None

        root = ET.fromstring(resp.content)
        body = root.find(".//body")
        if body is None:
            return None

        paragraphs = []
        for elem in body.iter():
            if elem.tag in {"p", "title", "sec"}:
                text = "".join(elem.itertext()).strip()
                if text:
                    paragraphs.append(text)

        full_text = "\n\n".join(paragraphs)
        return full_text if full_text else None

    except Exception:
        return None


def fetch_pubmed_for_gene(
    gene: str,
    drug: str,
    start_year: int = 2010,
    end_year: int = 2026,
    email: str = "<NCBI email here>",
    api_key: Optional[str] = None,
    max_results: int = 5
) -> List[Dict[str, Any]]:
    """Fetch PubMed papers for a single gene with dynamic drug context"""

    try:
        drug_terms = f'"{drug}"[All Fields]'

        all_terms = (
            '"Leukemia, Lymphoid"[MeSH Terms] OR '
            '"Precursor Cell Lymphoblastic Leukemia-Lymphoma"[MeSH Terms] OR '
            '"acute lymphoblastic leukemia"[All Fields] OR '
            '"acute lymphocytic leukemia"[All Fields] OR '
            '("ALL"[All Fields] AND leukemia[All Fields]) OR '
            '"B-ALL"[All Fields] OR '
            '"T-ALL"[All Fields]'
        )

        query = (
            f'('
            f'({gene}[All Fields] AND ({all_terms})) OR '
            f'({gene}[All Fields] AND ({drug_terms}) AND ({all_terms}))'
            f') AND ({start_year}:{end_year}[PDAT])'
        )

        headers = {"User-Agent": f"AML_UniqueGeneResearch/1.0 (mailto:{email})"}

        # ESearch
        search_params = {
            "db": "pubmed",
            "term": query,
            "retmax": max_results,
            "sort": "relevance",
            "email": email,
            "tool": "AML_UniqueGeneResearch"
        }
        if api_key:
            search_params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi",
            params=search_params,
            headers=headers
        )
        if not response.ok:
            return []

        root = ET.fromstring(response.text)
        pmids = [elem.text for elem in root.findall(".//Id")]

        if not pmids:
            return []

        # EFetch (abstracts + metadata)
        fetch_params = {
            "db": "pubmed",
            "id": ",".join(pmids),
            "retmode": "xml",
            "email": email,
            "tool": "AML_UniqueGeneResearch"
        }
        if api_key:
            fetch_params["api_key"] = api_key

        fetch_response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=fetch_params,
            headers=headers
        )
        if not fetch_response.ok:
            return []

        papers = parse_pubmed_xml(fetch_response.text)

        # ── NEW: attempt full-text retrieval for papers that have a PMCID ──
        for paper in papers:
            pmcid = paper.get("pmcid")
            if pmcid:
                time.sleep(0.35)   # polite gap before the extra PMC call
                full_text = fetch_pmc_full_text(pmcid, email, api_key)
                paper["full_text"] = full_text if full_text else "Full text not available via PMC"
            else:
                paper["full_text"] = "Full text not available via PMC"

        return papers

    except Exception as e:
        return []


def fetch_pubmed_for_all_unique_genes(
    gene_sets: Dict[str, List[str]],
    drug: str,
    start_year: int = 2010,
    end_year: int = 2026,
    email: str = "justinseby1998@gmail.com",
    api_key: Optional[str] = None,
    max_results_per_gene: int = 5
) -> pd.DataFrame:
    """Process all unique genes one by one and fetch PubMed literature"""

    unique_genes, gene_to_communities = extract_unique_genes_with_communities(gene_sets)

    all_results = []
    genes_with_papers = 0

    for gene in tqdm(unique_genes, desc=f"  Genes", unit="gene", leave=False, ncols=100):
        try:
            papers = fetch_pubmed_for_gene(
                gene=gene,
                drug=drug,
                start_year=start_year,
                end_year=end_year,
                email=email,
                api_key=api_key,
                max_results=max_results_per_gene
            )

            if papers:
                genes_with_papers += 1
                for paper in papers:
                    paper['gene']            = gene
                    paper['communities']     = ", ".join(gene_to_communities[gene])
                    paper['community_count'] = len(gene_to_communities[gene])
                    all_results.append(paper)

            time.sleep(0.3)  # be polite to NCBI

        except Exception as e:
            print(f"  Error processing gene {gene}: {str(e)}")
            continue

    print(f"  Genes with papers: {genes_with_papers}/{len(unique_genes)} "
          f"({genes_with_papers/len(unique_genes)*100:.1f}%)")
    print(f"  Total papers collected: {len(all_results)}")

    if not all_results:
        return pd.DataFrame()

    df = pd.DataFrame(all_results)

    column_order = [
        'gene', 'communities', 'community_count',
        'title', 'authors', 'journal', 'year',
        'volume', 'pages', 'doi', 'pmid', 'pmcid',
        'abstract', 'full_text',
    ]
    df = df[[col for col in column_order if col in df.columns]]

    return df


def parse_pubmed_xml(xml_content: str) -> List[Dict[str, Any]]:
    """Parse PubMed XML response to extract article details"""
    papers = []
    try:
        root = ET.fromstring(xml_content)
        for article_elem in root.findall(".//PubmedArticle"):
            try:
                paper = {}

                pmid_elem = article_elem.find(".//PMID")
                paper["pmid"] = pmid_elem.text if pmid_elem is not None else "Unknown"

                                # Title
                title_elem = article_elem.find(".//ArticleTitle")

                if title_elem is not None:
                    title_text = "".join(title_elem.itertext()).strip()
                    paper["title"] = title_text if title_text else "Unknown Title"
                else:
                    paper["title"] = "Unknown Title"

                abstract_parts = article_elem.findall(".//AbstractText")
                abstract_text  = " ".join([p.text for p in abstract_parts if p.text])
                paper["abstract"] = abstract_text if abstract_text else "Abstract not available"

                journal_elem = article_elem.find(".//Journal/Title")
                paper["journal"] = journal_elem.text if journal_elem is not None else "Unknown Journal"

                pub_date = article_elem.find(".//PubDate/Year")
                if pub_date is not None:
                    paper["year"] = pub_date.text
                else:
                    medline_date = article_elem.find(".//PubDate/MedlineDate")
                    if medline_date is not None and medline_date.text:
                        year_match = re.search(r'\d{4}', medline_date.text)
                        paper["year"] = year_match.group(0) if year_match else "Unknown"
                    else:
                        paper["year"] = "Unknown"

                volume_elem = article_elem.find(".//Volume")
                paper["volume"] = volume_elem.text if volume_elem is not None else "Unknown"

                pages_elem = article_elem.find(".//MedlinePgn")
                paper["pages"] = pages_elem.text if pages_elem is not None else "Unknown"

                authors = []
                author_list = article_elem.find(".//AuthorList")
                if author_list is not None:
                    for author in author_list.findall(".//Author"):
                        last_name = author.find("LastName")
                        initials  = author.find("Initials")
                        if last_name is not None and initials is not None:
                            authors.append(f"{last_name.text} {initials.text}")
                        elif last_name is not None:
                            authors.append(last_name.text)
                paper["authors"] = ", ".join(authors) if authors else "Unknown Authors"

                doi, pmc_id = None, None
                article_id_list = article_elem.find(".//ArticleIdList")
                if article_id_list is not None:
                    for id_elem in article_id_list.findall(".//ArticleId"):
                        if id_elem.get("IdType") == "doi":
                            doi = id_elem.text
                        elif id_elem.get("IdType") == "pmc":
                            pmc_id = id_elem.text

                paper["doi"]   = doi if doi else "DOI not available"
                paper["pmcid"] = pmc_id if pmc_id else None

                papers.append(paper)

            except Exception:
                continue

    except Exception:
        pass

    return papers


# ── MAIN ORCHESTRATOR ─────────────────────────────────────────────────────────

if __name__ == "__main__":

    ANNOTATION_CSV       = "DrugAnnotation.csv"
    OUTPUT_FOLDER        = "papers_per_drug"
    EMAIL                = "< NCBI Email >"
    API_KEY              = "< NCBI API Key >"
    START_YEAR           = 2015
    END_YEAR             = 2026
    MAX_RESULTS_PER_GENE = 50

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    annotation_df = pd.read_csv(ANNOTATION_CSV)
    print(f"Loaded {len(annotation_df)} rows from {ANNOTATION_CSV}\n")

    for _, row in tqdm(annotation_df.iterrows(), total=len(annotation_df),
                   desc="Overall progress", unit="drug", ncols=100):
        drug_direction = str(row["Community"])
        drug           = str(row["Drug"])
        direction      = str(row["Direction"])

        output_csv = os.path.join(OUTPUT_FOLDER, f"{drug_direction}_Papers.csv")
        if os.path.exists(output_csv):
            print(f"Already done, skipping: {drug_direction}")
            continue

        print(f"\n{'='*55}")
        print(f"Processing : {drug_direction}  (drug={drug}, direction={direction})")
        print(f"{'='*55}")
        genes = [g.strip() for g in str(row["Genes_String"]).split(",") if g.strip()]
        if not genes:
            print(f"  No genes found for {drug_direction}, skipping.")
            pd.DataFrame().to_csv(output_csv, index=False)
            continue

        gene_sets = {drug_direction: genes}
        papers_df = fetch_pubmed_for_all_unique_genes(
            gene_sets=gene_sets,
            drug=drug,
            start_year=START_YEAR,
            end_year=END_YEAR,
            email=EMAIL,
            api_key=API_KEY,
            max_results_per_gene=MAX_RESULTS_PER_GENE
        )

        if papers_df.empty:
            print(f"  No papers found for {drug_direction}")
            pd.DataFrame().to_csv(output_csv, index=False)
        else:
            papers_df.to_csv(output_csv, index=False)
            print(f"  Saved {len(papers_df)} papers -> {output_csv}")

    print(f"\n{'='*55}")
    print(f"All done! Paper CSVs saved in: {OUTPUT_FOLDER}/")
    print(f"{'='*55}")

Loaded 170 rows from DrugAnnotation.csv



Overall progress:   0%|                                                   | 0/170 [00:00<?, ?drug/s]

Already done, skipping: A-1155463_UP
Already done, skipping: A-1331852_UP
Already done, skipping: A-419259_UP
Already done, skipping: Abemaciclib_UP
Already done, skipping: Abexinostat_UP
Already done, skipping: Aldoxorubicin_UP
Already done, skipping: Alisertib_UP
Already done, skipping: Altiratinib_UP
Already done, skipping: AMG-232_UP
Already done, skipping: AMG-925_UP
Already done, skipping: AMG319_UP
Already done, skipping: Asciminib_UP
Already done, skipping: AT-101_UP
Already done, skipping: AT9283_UP
Already done, skipping: AVN944_UP
Already done, skipping: Azacitidine_UP
Already done, skipping: AZD-5363_UP
Already done, skipping: AZD-8186_UP
Already done, skipping: AZD1152-HQPA_UP
Already done, skipping: AZD1208_UP
Already done, skipping: AZD6738_UP
Already done, skipping: AZD7762_UP
Already done, skipping: AZD8055_UP
Already done, skipping: BI-2536_UP
Already done, skipping: BIIB021_UP
Already done, skipping: Binimetinib_UP
Already done, skipping: Birabresib_UP
Already done, 


Overall progress:  39%|████████████████▎                         | 66/170 [05:15<08:17,  4.79s/drug]

  Genes with papers: 36/140 (25.7%)
  Total papers collected: 301
  Saved 301 papers -> papers_per_drug/Fimepinostat (CUDC-907)_UP_Papers.csv

Processing : Foretinib_UP  (drug=Foretinib, direction=UP)
  Extracted 64 unique genes from 1 gene sets


...

  Saved 751 papers -> papers_per_drug/WEHI-539_UP_Papers.csv

Processing : ZSTK474_UP  (drug=ZSTK474, direction=UP)
  Extracted 131 unique genes from 1 gene sets



  Genes: 100%|██████████████████████████████████████████████████| 131/131 [07:29<00:00,  1.21s/gene]
                                                                                                    

  Genes with papers: 55/131 (42.0%)
  Total papers collected: 610


Overall progress: 100%|██████████████████████████████████████| 170/170 [5:48:37<00:00, 123.05s/drug]

  Saved 610 papers -> papers_per_drug/ZSTK474_UP_Papers.csv

All done! Paper CSVs saved in: papers_per_drug/
